# Agent Memory Frameworks

**Module:** 13 — AI Memory

LangChain, LangGraph, LlamaIndex, MemGPT/Letta, Zep — patterns and comparison.

---


## How to Use This Notebook

1. Read each major section fully before running code — the narrative carries the design intuition.
2. Run code cells top-to-bottom. They are self-contained demos (stdlib / light mocks) unless noted.
3. Treat API keys as **placeholders**: set `OPENAI_API_KEY`, `ANTHROPIC_API_KEY`, etc. in your environment — never hardcode secrets.
4. Complete **Try it yourself** exercises; they are the difference between recognition and skill.


## Learning Objectives

By the end of this notebook, you will be able to:

- Describe how major frameworks expose memory APIs
- Contrast buffer, summary, vector, and store-based memories
- Pick a framework approach for a given product constraint
- Sketch integration code shapes (with placeholders)


## Landscape overview

| Framework | Memory style | Strength | Watch-out |
|-----------|--------------|----------|-----------|
| LangChain | Buffer/summary/vector classes | Fast to prototype | Easy to outgrow abstractions |
| LangGraph | Checkpointed state + stores | Durable agent runs | You design the schema |
| LlamaIndex | Chat stores + vector indices | RAG-native | Memory vs doc index blur |
| MemGPT / Letta | OS-like paging in/out | Long-horizon agents | Operational complexity |
| Zep | Managed memory service | Session + knowledge APIs | Vendor coupling |

```mermaid
flowchart LR
  App --> FW[Framework memory API]
  FW --> STM[Session buffer]
  FW --> VEC[Vector store]
  FW --> CHK[Checkpoints]
```


## LangChain Memory

### Definition
Classic chat memory wrappers: buffer, window buffer, summary, and vector-backed entity memories that attach to chains/agents.

### Why it matters
Still the most common teaching surface; many apps started here.

### How it works
Messages append to a memory object; before each call, `load_memory_variables` injects history; after, `save_context` persists.

### Intuition
A notepad clipped onto the chain — simple, sometimes too opaque.

### Pitfalls
- Summary memory hallucinating history
- Global memory across users (missing session keys)
- Hiding persistence details until production pain

### When to use
Prototypes and simple chatbots; migrate hot paths to explicit stores as you scale.


In [ ]:
# Demo 1: LangChain-like buffer memory interface (educational mock)
class BufferMemory:
    def __init__(self):
        self.chat_memory = []  # list[tuple[str,str]]

    def load_memory_variables(self, inputs):
        history = "\n".join(f"{r}: {c}" for r, c in self.chat_memory)
        return {"history": history}

    def save_context(self, inputs, outputs):
        self.chat_memory.append(("user", inputs["input"]))
        self.chat_memory.append(("assistant", outputs["output"]))

mem = BufferMemory()
mem.save_context({"input": "Hi"}, {"output": "Hello!"})
mem.save_context({"input": "I like cats"}, {"output": "Noted."})
print(mem.load_memory_variables({}))


In [ ]:
# Demo 2: window vs summary policy
class WindowMemory(BufferMemory):
    def __init__(self, k=2):
        super().__init__()
        self.k = k  # last k *pairs*

    def load_memory_variables(self, inputs):
        pairs = self.chat_memory[-(2 * self.k):]
        history = "\n".join(f"{r}: {c}" for r, c in pairs)
        return {"history": history}

wm = WindowMemory(k=1)
for i in range(3):
    wm.save_context({"input": f"u{i}"}, {"output": f"a{i}"})
print(wm.load_memory_variables({}))


## LangGraph Memory

### Definition
Graph-shaped agents with **checkpointed state** (threads) and optional long-term stores — memory as explicit state channels.

### Why it matters
Production agents need resume, time-travel debug, and durable state; LangGraph centers that.

### How it works
Define state schema → nodes read/write state → checkpointer persists per `thread_id` → optional store for cross-thread memories.

### Intuition
The agent’s hard drive is the checkpoint; the sticky notes across chats are the store.

### Pitfalls
- Putting huge blobs in state every step
- Forgetting to key by thread/user
- No schema migrations for state

### When to use
Multi-step agents, HITL, long-running workflows.


In [ ]:
# Demo 3: checkpoint dict keyed by thread_id
from copy import deepcopy

class Checkpointer:
    def __init__(self):
        self._threads = {}

    def put(self, thread_id: str, state: dict):
        self._threads[thread_id] = deepcopy(state)

    def get(self, thread_id: str) -> dict | None:
        return deepcopy(self._threads.get(thread_id))

cp = Checkpointer()
state = {"messages": ["hi"], "prefs": {}}
cp.put("thread-1", state)
state["messages"].append("I like dark mode")
# not saved yet
print("disk", cp.get("thread-1"))
cp.put("thread-1", state)
print("disk after", cp.get("thread-1"))


## LlamaIndex Memory

### Definition
Chat memory modules plus first-class indices — memory often overlaps with RAG retrieval abstractions.

### Why it matters
If your agent is document-heavy, one stack for chat history + retrieval reduces glue code.

### How it works
Use chat stores for turn history; vector/KG indices for knowledge; composable retrievers feed the chat engine.

### Intuition
Library + conversation log under one roof.

### Pitfalls
- Dumping chat into the same index as docs without type tags
- Unclear retention for chat vs knowledge

### When to use
RAG assistants that also need conversational continuity.


In [ ]:
# Demo 4: chat store + knowledge index separation
chat_store = {"u_42": [("user", "Explain invoice #9"), ("assistant", "..." )]}
knowledge_index = {"inv_policy": "Refunds within 30 days", "tax": "VAT 20%"}

def retrieve_context(user_id: str, query: str):
    history = chat_store.get(user_id, [])[-4:]
    docs = [v for k, v in knowledge_index.items() if k.split("_")[0] in query or True]
    return {"history": history, "docs": docs[:2]}

print(retrieve_context("u_42", "invoice refund"))


## MemGPT / Letta

### Definition
Agents that actively page information in/out of a limited context — self-editing memory inspired by OS virtual memory.

### Why it matters
Long-horizon agents hit context limits; paging is a principled strategy.

### How it works
Core/main context holds working set; archival memory is external; the agent emits memory edit actions.

### Intuition
The model is told it has a small desk and a big filing room — and tools to move papers.

### Pitfalls
- Unbounded archival growth
- Agent forgetting to write critical facts out
- Harder eval surface

### When to use
Research agents, personal OS agents, multi-hour tasks.


In [ ]:
# Demo 5: tiny paging policy
MAIN_LIMIT = 3
main_ctx = []
archival = []

def observe(event: str):
    main_ctx.append(event)
    while len(main_ctx) > MAIN_LIMIT:
        archival.append(main_ctx.pop(0))

for e in ["goal: write PR", "file: a.py", "file: b.py", "test failed", "fix ok"]:
    observe(e)
print("main", main_ctx)
print("archival", archival)


## Zep

### Definition
A managed memory service exposing session history, summarization, and knowledge extraction APIs for assistants.

### Why it matters
Teams that want memory without operating vector/SQL infra use hosted memory.

### How it works
Create session → append messages → query memory / facts; integrate via REST/SDK.

### Intuition
Memory as a SaaS dependency — fast path, vendor trade-offs.

### Pitfalls
- Data residency constraints
- Coupling eval to vendor features

### When to use
Product teams prioritizing speed + managed ops over deep customization.


In [ ]:
# Demo 6: Zep-like REST shapes (placeholders)
import os, json
ZEP_API_KEY = os.getenv("ZEP_API_KEY", "YOUR_ZEP_API_KEY")
add_memory_request = {
    "session_id": "sess_123",
    "messages": [
        {"role": "user", "content": "I am allergic to peanuts"},
        {"role": "assistant", "content": "I will remember that."},
    ],
}
search_request = {"session_id": "sess_123", "text": "dietary restrictions", "limit": 5}
fake_response = {
    "results": [{"message": {"content": "I am allergic to peanuts"}, "score": 0.92}],
}
print(json.dumps({"add": add_memory_request, "search": search_request, "resp": fake_response}, indent=2))
print("key placeholder?", ZEP_API_KEY.startswith("YOUR_"))


## Framework Comparison

| Criterion | Prefer |
|-----------|--------|
| Fast chat prototype | LangChain buffer/summary |
| Durable multi-actor workflows | LangGraph checkpoints + store |
| Doc-heavy RAG assistant | LlamaIndex |
| Autonomous long-horizon paging | Letta/MemGPT-style |
| Managed memory SaaS | Zep (or similar) |
| Strict multi-tenant enterprise | Often **custom** on Postgres+vector |

### Decision micro-flow
```
Need durable resume? → LangGraph/custom checkpoints
Need semantic recall? → Vector store (any stack)
Need managed ops? → Zep-like service
Need max control/compliance? → Custom memory service
```


### Try it yourself — Framework choice

1. Pick a framework for: (a) HIPAA chatbot, (b) weekend hackathon bot, (c) week-long research agent — justify in 3 bullets each.
2. Extend the checkpointer mock with `list_versions(thread_id)`.


## Glossary / Key Terms

| Term | Meaning |
|------|---------|
| `checkpointer` | Persists graph/agent state per thread |
| `chat store` | Storage for conversational messages |
| `archival memory` | Long-term store outside main context (MemGPT-style) |
| `memory variables` | LangChain dict injected into prompts |


## Integration Boundary Pattern (Recommended)

Even if you start with LangChain/LangGraph memory classes, put a **MemoryService** interface in front:

```python
class MemoryService(Protocol):
    def retrieve(self, user_id: str, query: str, k: int) -> list[Memory]: ...
    def upsert(self, user_id: str, item: Memory) -> str: ...
    def forget(self, user_id: str, mem_id: str) -> None: ...
```

Frameworks become adapters. Swapping Zep → custom Postgres+Qdrant won't rewrite your agent.


In [ ]:
# Adapter pattern sketch
class MemoryService:
    def retrieve(self, user_id, query, k=5):
        raise NotImplementedError

class LangChainBufferAdapter(MemoryService):
    def __init__(self):
        self.buf = []
    def retrieve(self, user_id, query, k=5):
        return self.buf[-k:]
    def upsert(self, user_id, item):
        self.buf.append(item)
        return str(len(self.buf))

a = LangChainBufferAdapter()
a.upsert("u", "hi")
print(a.retrieve("u", "q"))


In [ ]:
# Feature comparison scoring for a HIPAA chatbot
weights = {"self_host": 0.35, "audit": 0.25, "hitl": 0.1, "speed": 0.15, "cost": 0.15}
options = {
    "langchain_buffer": {"self_host": 1, "audit": 0.3, "hitl": 0.4, "speed": 0.9, "cost": 0.9},
    "langgraph_store": {"self_host": 1, "audit": 0.7, "hitl": 0.8, "speed": 0.6, "cost": 0.7},
    "zep_cloud": {"self_host": 0.2, "audit": 0.6, "hitl": 0.5, "speed": 0.8, "cost": 0.5},
    "custom": {"self_host": 1, "audit": 1, "hitl": 0.9, "speed": 0.5, "cost": 0.4},
}

def score(caps):
    return sum(weights[k] * caps[k] for k in weights)

print(sorted(((n, round(score(c), 3)) for n, c in options.items()), key=lambda x: -x[1]))


### Try it yourself — Frameworks deepen

1. Write a 10-line ADR: why custom MemoryService for multi-tenant SaaS.
2. Map Letta archival memory to your Module 13 lifecycle stages.


## Key Takeaways

- Frameworks differ more in state/durability models than in embeddings
- Always key memory by user/session/tenant — frameworks won't save you if you don't
- Prototype fast, then extract a clear memory service boundary
- Evaluate with your retrieval metrics, not vendor demos alone
